# DTP Managed Roads — Gold Asset Intelligence

## Purpose

Transform the analysis-ready Silver road-segment dataset into business-facing
Asset Intelligence metrics and reporting tables for Power BI.

## Business Questions

This Gold layer will support questions such as:

- How many managed-road segments are represented?
- What is the total represented managed-road network length?
- How is network length distributed by road classification?
- How is network length distributed by RMA class?
- Which localities contain the largest represented managed-road network?
- Which road types contribute the greatest network length?
- What proportion of records pass the Silver data-quality checks?
- Where are the main attribute-quality exceptions?

In [0]:
%pip install geopandas pyogrio

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import geopandas as gpd
import pandas as pd
import numpy as np

print("Gold libraries loaded successfully")

Gold libraries loaded successfully


## Load Silver Dataset

Load the cleaned and enriched Silver Managed Roads dataset.

Gold consumes Silver only; it does not return to the original published source.

In [0]:
silver_path = "/Volumes/dtp_data/dtp_schema/dtp_geojson/managed_roads_silver.geojson"

roads_silver = gpd.read_file(silver_path)

print("=== SILVER INPUT ===")
print(f"Records : {len(roads_silver):,}")
print(f"Columns : {len(roads_silver.columns)}")
print(f"CRS     : {roads_silver.crs}")

=== SILVER INPUT ===
Records : 90,797
Columns : 35
CRS     : EPSG:4326


In [0]:
required_cols = [
    "OBJECTID",
    "RD_NAME",
    "RD_TYPE",
    "CLASSN",
    "RMACLASS",
    "LOCALITY",
    "SEGMENT_LENGTH_M",
    "SEGMENT_LENGTH_KM",
    "DQ_OVERALL_STATUS",
    "geometry"
]

missing_required_cols = [
    col for col in required_cols
    if col not in roads_silver.columns
]

assert len(missing_required_cols) == 0, \
    f"Required Gold fields missing: {missing_required_cols}"

assert roads_silver["OBJECTID"].duplicated().sum() == 0, \
    "Duplicate OBJECTIDs found in Silver"

assert roads_silver["SEGMENT_LENGTH_KM"].isna().sum() == 0, \
    "Missing segment lengths detected"

assert (roads_silver["SEGMENT_LENGTH_KM"] < 0).sum() == 0, \
    "Negative segment lengths detected"

print("Gold input validation: PASS")


Gold input validation: PASS


## Gold KPI Definitions

Gold uses segment count and derived segment length as the principal network
measures.

Network length represents the summed length of the managed-road geometries
contained in this dataset.

Data-quality status is inherited from Silver and is not recalculated in Gold.


In [0]:
gold_network_overview = pd.DataFrame({
    "metric": [
        "Road segments",
        "Represented network length (km)",
        "Average segment length (km)",
        "Median segment length (km)",
        "Localities represented",
        "Road names represented"
    ],
    "value": [
        len(roads_silver),

        round(
            roads_silver["SEGMENT_LENGTH_KM"].sum(),
            2
        ),

        round(
            roads_silver["SEGMENT_LENGTH_KM"].mean(),
            3
        ),

        round(
            roads_silver["SEGMENT_LENGTH_KM"].median(),
            3
        ),

        roads_silver["LOCALITY"].nunique(
            dropna=True
        ),

        roads_silver["RD_NAME"].nunique(
            dropna=True
        )
    ]
})

display(gold_network_overview)

metric,value
Road segments,90797.0
Represented network length (km),26297.46
Average segment length (km),0.29
Median segment length (km),0.133
Localities represented,2219.0
Road names represented,886.0


## Network by Road Classification

Summarise segment count and represented network length by `CLASSN`.

Source classification codes are retained without inferring descriptions that
are not provided by authoritative metadata.

In [0]:
gold_network_by_class = (
    roads_silver
    .groupby(
        "CLASSN",
        dropna=False
    )
    .agg(
        segment_count=(
            "OBJECTID",
            "count"
        ),

        network_length_km=(
            "SEGMENT_LENGTH_KM",
            "sum"
        ),

        average_segment_length_km=(
            "SEGMENT_LENGTH_KM",
            "mean"
        )
    )
    .reset_index()
)

gold_network_by_class[
    "network_length_km"
] = gold_network_by_class[
    "network_length_km"
].round(2)

gold_network_by_class[
    "average_segment_length_km"
] = gold_network_by_class[
    "average_segment_length_km"
].round(3)

total_km = roads_silver[
    "SEGMENT_LENGTH_KM"
].sum()

gold_network_by_class[
    "network_pct"
] = (
    gold_network_by_class[
        "network_length_km"
    ]
    / total_km
    * 100
).round(2)

gold_network_by_class = (
    gold_network_by_class
    .sort_values(
        "network_length_km",
        ascending=False
    )
    .reset_index(drop=True)
)

display(gold_network_by_class)

CLASSN,segment_count,network_length_km,average_segment_length_km,network_pct
MR,52584,13949.85,0.265,53.05
HW,25696,7638.84,0.297,29.05
FW,7405,2625.26,0.355,9.98
TR,4350,1683.62,0.387,6.4
FR,572,356.13,0.623,1.35
NR,177,31.45,0.178,0.12
PR,13,12.32,0.947,0.05


In [0]:
gold_network_by_rma_class = (
    roads_silver
    .groupby(
        "RMACLASS",
        dropna=False
    )
    .agg(
        segment_count=(
            "OBJECTID",
            "count"
        ),

        network_length_km=(
            "SEGMENT_LENGTH_KM",
            "sum"
        )
    )
    .reset_index()
)

gold_network_by_rma_class[
    "network_length_km"
] = gold_network_by_rma_class[
    "network_length_km"
].round(2)

gold_network_by_rma_class[
    "network_pct"
] = (
    gold_network_by_rma_class[
        "network_length_km"
    ]
    / total_km
    * 100
).round(2)

gold_network_by_rma_class = (
    gold_network_by_rma_class
    .sort_values(
        "network_length_km",
        ascending=False
    )
    .reset_index(drop=True)
)

display(gold_network_by_rma_class)

RMACLASS,segment_count,network_length_km,network_pct
AO,56178,15670.85,59.59
AH,27032,7960.76,30.27
FW,7397,2622.09,9.97
NR,177,31.45,0.12
PR,13,12.32,0.05


## Network by Road Type

Identify the road types contributing the greatest represented network length.

In [0]:
gold_network_by_road_type = (
    roads_silver
    .groupby(
        "RD_TYPE",
        dropna=False
    )
    .agg(
        segment_count=(
            "OBJECTID",
            "count"
        ),

        network_length_km=(
            "SEGMENT_LENGTH_KM",
            "sum"
        )
    )
    .reset_index()
)

gold_network_by_road_type[
    "network_length_km"
] = gold_network_by_road_type[
    "network_length_km"
].round(2)

gold_network_by_road_type[
    "network_pct"
] = (
    gold_network_by_road_type[
        "network_length_km"
    ]
    / total_km
    * 100
).round(2)

gold_network_by_road_type = (
    gold_network_by_road_type
    .sort_values(
        "network_length_km",
        ascending=False
    )
    .reset_index(drop=True)
)

display(gold_network_by_road_type)

RD_TYPE,segment_count,network_length_km,network_pct
ROAD,54589,15736.63,59.84
HIGHWAY,21606,6542.14,24.88
FREEWAY,4532,1848.92,7.03
HIGHWAY EAST,2301,575.01,2.19
HIGHWAY WEST,1572,470.52,1.79
FREEWAY EAST,579,251.48,0.96
null,1321,213.96,0.81
FREEWAY WEST,534,205.79,0.78
STREET,2144,187.35,0.71
WAY,193,56.23,0.21


## Network by Locality

Summarise represented managed-road network length by locality to support
geographic comparison.


In [0]:
gold_network_by_locality = (
    roads_silver
    .groupby(
        "LOCALITY",
        dropna=False
    )
    .agg(
        segment_count=(
            "OBJECTID",
            "count"
        ),

        network_length_km=(
            "SEGMENT_LENGTH_KM",
            "sum"
        ),

        road_count=(
            "RD_NAME",
            "nunique"
        )
    )
    .reset_index()
)

gold_network_by_locality[
    "network_length_km"
] = gold_network_by_locality[
    "network_length_km"
].round(2)

gold_network_by_locality = (
    gold_network_by_locality
    .sort_values(
        "network_length_km",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    gold_network_by_locality.head(30)
)

LOCALITY,segment_count,network_length_km,road_count
OUYEN,137,103.9,4
NHILL,196,100.13,5
BENALLA,312,94.15,6
HOPETOUN,122,82.27,4
NARIEL VALLEY,87,80.53,1
MITTA MITTA,52,66.61,2
WERRIBEE,385,65.11,8
WARRACKNABEAL,130,63.87,5
SUNBURY,346,63.84,6
RAINBOW,74,58.9,4


In [0]:
gold_network_by_road = (
    roads_silver
    .groupby(
        [
            "RD_NAME",
            "RD_TYPE"
        ],
        dropna=False
    )
    .agg(
        segment_count=(
            "OBJECTID",
            "count"
        ),

        network_length_km=(
            "SEGMENT_LENGTH_KM",
            "sum"
        ),

        locality_count=(
            "LOCALITY",
            "nunique"
        )
    )
    .reset_index()
)

gold_network_by_road[
    "network_length_km"
] = gold_network_by_road[
    "network_length_km"
].round(2)

gold_network_by_road = (
    gold_network_by_road
    .sort_values(
        "network_length_km",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    gold_network_by_road.head(30)
)

RD_NAME,RD_TYPE,segment_count,network_length_km,locality_count
HUME,FREEWAY,1138,661.07,55
MURRAY VALLEY,HIGHWAY,1776,647.16,76
PRINCES,HIGHWAY EAST,2301,575.01,85
PRINCES,HIGHWAY WEST,1572,470.52,57
CALDER,HIGHWAY,1027,463.4,48
MIDLAND,HIGHWAY,1754,448.27,75
HENTY,HIGHWAY,748,344.03,35
SUNRAYSIA,HIGHWAY,741,338.92,37
WESTERN,FREEWAY,730,324.13,43
WESTERN,HIGHWAY,789,321.74,37


## Data Quality Performance

Summarise Silver quality indicators so stakeholders can understand both the
network information and its data-quality limitations.

In [0]:
gold_data_quality_status = (
    roads_silver[
        "DQ_OVERALL_STATUS"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "DQ_OVERALL_STATUS"
    )
    .reset_index(
        name="record_count"
    )
)

gold_data_quality_status[
    "record_pct"
] = (
    gold_data_quality_status[
        "record_count"
    ]
    / len(roads_silver)
    * 100
).round(2)

display(gold_data_quality_status)

DQ_OVERALL_STATUS,record_count,record_pct
COMPLETE,89034,98.06
REVIEW,1763,1.94


In [0]:
gold_data_quality_summary = pd.DataFrame({
    "quality_metric": [
        "Total records",
        "Records complete",
        "Records requiring review",
        "Missing DEC_TYPE",
        "Missing RD_TYPE",
        "Missing LOCAL_TYPE",
        "Zero-length segments",
        "Invalid geometries"
    ],

    "record_count": [
        len(roads_silver),

        (
            roads_silver[
                "DQ_OVERALL_STATUS"
            ] == "COMPLETE"
        ).sum(),

        (
            roads_silver[
                "DQ_OVERALL_STATUS"
            ] == "REVIEW"
        ).sum(),

        roads_silver[
            "DQ_MISSING_DEC_TYPE"
        ].sum(),

        roads_silver[
            "DQ_MISSING_RD_TYPE"
        ].sum(),

        roads_silver[
            "DQ_MISSING_LOCAL_TYPE"
        ].sum(),

        roads_silver[
            "DQ_ZERO_LENGTH"
        ].sum(),

        roads_silver[
            "DQ_INVALID_GEOMETRY"
        ].sum()
    ]
})

gold_data_quality_summary[
    "record_pct"
] = (
    gold_data_quality_summary[
        "record_count"
    ]
    / len(roads_silver)
    * 100
).round(2)

display(gold_data_quality_summary)

quality_metric,record_count,record_pct
Total records,90797,100.0
Records complete,89034,98.06
Records requiring review,1763,1.94
Missing DEC_TYPE,1321,1.45
Missing RD_TYPE,1321,1.45
Missing LOCAL_TYPE,991,1.09
Zero-length segments,0,0.0
Invalid geometries,0,0.0


In [0]:
gold_dq_by_locality = (
    roads_silver
    .groupby(
        "LOCALITY",
        dropna=False
    )
    .agg(
        total_segments=(
            "OBJECTID",
            "count"
        ),

        review_segments=(
            "DQ_ANY_ISSUE",
            "sum"
        )
    )
    .reset_index()
)

gold_dq_by_locality[
    "review_pct"
] = (
    gold_dq_by_locality[
        "review_segments"
    ]
    / gold_dq_by_locality[
        "total_segments"
    ]
    * 100
).round(2)

gold_dq_by_locality = (
    gold_dq_by_locality
    .sort_values(
        [
            "review_pct",
            "review_segments"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

display(
    gold_dq_by_locality.head(30)
)

LOCALITY,total_segments,review_segments,review_pct
INDENTED HEAD,27,18,66.67
LAKES ENTRANCE,38,24,63.16
WEST MELBOURNE,422,225,53.32
MOUNT MARTHA,106,51,48.11
CREMORNE,47,22,46.81
PASCOE VALE SOUTH,99,42,42.42
BURNLEY,75,29,38.67
WOODEND NORTH,26,10,38.46
STRATHMORE,116,44,37.93
BRUNSWICK WEST,86,32,37.21


## Power BI Fact Table

Create a business-facing road-segment fact table for detailed Power BI
filtering and drill-down.

Gold aggregations provide KPI tables, while this detailed fact table supports
interactive analysis.


In [0]:
gold_managed_roads_fact = roads_silver[
    [
        "OBJECTID",

        "RD_NAME",
        "RD_TYPE",

        "DEC_NAME",
        "DEC_TYPE",

        "LOCAL_NAME",
        "LOCAL_TYPE",

        "CLASSN",
        "RMACLASS",

        "RD_NUM",
        "RD_SECTION",

        "PROFILE",
        "SRNS",

        "RMANUM",
        "LOCALITY",

        "SEGMENT_LENGTH_M",
        "SEGMENT_LENGTH_KM",

        "DQ_MISSING_DEC_TYPE",
        "DQ_MISSING_RD_TYPE",
        "DQ_MISSING_LOCAL_TYPE",
        "DQ_CORE_ATTRIBUTE_ISSUE",

        "DQ_ZERO_LENGTH",
        "DQ_INVALID_GEOMETRY",
        "DQ_ANY_ISSUE",
        "DQ_OVERALL_STATUS"
    ]
].copy()

print(
    f"Gold fact records : {len(gold_managed_roads_fact):,}"
)

print(
    f"Gold fact columns : {len(gold_managed_roads_fact.columns)}"
)

display(
    gold_managed_roads_fact.head(10)
)

Gold fact records : 90,797
Gold fact columns : 25


OBJECTID,RD_NAME,RD_TYPE,DEC_NAME,DEC_TYPE,LOCAL_NAME,LOCAL_TYPE,CLASSN,RMACLASS,RD_NUM,RD_SECTION,PROFILE,SRNS,RMANUM,LOCALITY,SEGMENT_LENGTH_M,SEGMENT_LENGTH_KM,DQ_MISSING_DEC_TYPE,DQ_MISSING_RD_TYPE,DQ_MISSING_LOCAL_TYPE,DQ_CORE_ATTRIBUTE_ISSUE,DQ_ZERO_LENGTH,DQ_INVALID_GEOMETRY,DQ_ANY_ISSUE,DQ_OVERALL_STATUS
1,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,MR,AO,5248,01,1,N,5248,WEST MELBOURNE,34.740411898025485,0.034740411898025486,false,false,false,false,false,false,false,COMPLETE
2,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,MR,AO,5248,01,8,N,5248,WEST MELBOURNE,44.767868208077275,0.04476786820807727,false,false,false,false,false,false,false,COMPLETE
3,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,MR,AO,5248,01,8,N,5248,WEST MELBOURNE,52.7438736000815,0.0527438736000815,false,false,false,false,false,false,false,COMPLETE
4,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,MR,AO,5248,01,8,N,5248,WEST MELBOURNE,19.106245845077993,0.019106245845077995,false,false,false,false,false,false,false,COMPLETE
5,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,MR,AO,5248,01,8,N,5248,WEST MELBOURNE,10.693296337946041,0.01069329633794604,false,false,false,false,false,false,false,COMPLETE
6,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,MR,AO,5248,01,8,N,5248,WEST MELBOURNE,81.83874593055961,0.0818387459305596,false,false,false,false,false,false,false,COMPLETE
7,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,MR,AO,5248,01,1,N,5248,WEST MELBOURNE,55.555896340076586,0.055555896340076585,false,false,false,false,false,false,false,COMPLETE
8,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,MR,AO,5248,01,8,N,5248,WEST MELBOURNE,48.23820120495433,0.048238201204954326,false,false,false,false,false,false,false,COMPLETE
9,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,MR,AO,5248,01,8,N,5248,WEST MELBOURNE,58.30401234522537,0.058304012345225364,false,false,false,false,false,false,false,COMPLETE
10,MACKENZIE,ROAD,MACKENZIE,ROAD,MACKENZIE,ROAD,MR,AO,5248,01,1,N,5248,WEST MELBOURNE,34.03908455013818,0.034039084550138175,false,false,false,false,false,false,false,COMPLETE


In [0]:
print("=== GOLD CONSISTENCY CHECKS ===")

fact_total_km = (
    gold_managed_roads_fact[
        "SEGMENT_LENGTH_KM"
    ].sum()
)

class_total_km = (
    gold_network_by_class[
        "network_length_km"
    ].sum()
)

print(
    f"Silver total km : {total_km:,.2f}"
)

print(
    f"Fact total km   : {fact_total_km:,.2f}"
)

print(
    f"Class total km  : {class_total_km:,.2f}"
)

assert len(
    gold_managed_roads_fact
) == len(roads_silver), \
    "Gold fact row count differs from Silver"

assert np.isclose(
    fact_total_km,
    total_km,
    rtol=0,
    atol=0.01
), "Gold fact network length differs from Silver"

print("\nGold consistency validation: PASS")

=== GOLD CONSISTENCY CHECKS ===
Silver total km : 26,297.46
Fact total km   : 26,297.46
Class total km  : 26,297.47

Gold consistency validation: PASS


## Persist Gold Outputs

Persist business-facing Gold datasets for downstream Power BI consumption.

In [0]:
gold_base_path = "/Volumes/dtp_data/dtp_schema/dtp_geojson/gold"

dbutils.fs.mkdirs(gold_base_path)

print("Gold output directory created:")
print(gold_base_path)

Gold output directory created:
/Volumes/dtp_data/dtp_schema/dtp_geojson/gold


In [0]:
gold_network_overview.to_csv(
    f"{gold_base_path}/gold_network_overview.csv",
    index=False
)

gold_network_by_class.to_csv(
    f"{gold_base_path}/gold_network_by_class.csv",
    index=False
)

gold_network_by_rma_class.to_csv(
    f"{gold_base_path}/gold_network_by_rma_class.csv",
    index=False
)

gold_network_by_road_type.to_csv(
    f"{gold_base_path}/gold_network_by_road_type.csv",
    index=False
)

gold_network_by_locality.to_csv(
    f"{gold_base_path}/gold_network_by_locality.csv",
    index=False
)

gold_network_by_road.to_csv(
    f"{gold_base_path}/gold_network_by_road.csv",
    index=False
)

gold_data_quality_summary.to_csv(
    f"{gold_base_path}/gold_data_quality_summary.csv",
    index=False
)

gold_dq_by_locality.to_csv(
    f"{gold_base_path}/gold_dq_by_locality.csv",
    index=False
)

gold_managed_roads_fact.to_csv(
    f"{gold_base_path}/gold_managed_roads_fact.csv",
    index=False
)

print("Gold CSV outputs saved successfully")

Gold CSV outputs saved successfully


In [0]:
spark.createDataFrame(
    gold_network_overview
).write.mode(
    "overwrite"
).saveAsTable(
    "dtp_data.dtp_schema.gold_network_overview"
)

spark.createDataFrame(
    gold_network_by_class
).write.mode(
    "overwrite"
).saveAsTable(
    "dtp_data.dtp_schema.gold_network_by_class"
)

spark.createDataFrame(
    gold_network_by_rma_class
).write.mode(
    "overwrite"
).saveAsTable(
    "dtp_data.dtp_schema.gold_network_by_rma_class"
)

spark.createDataFrame(
    gold_network_by_road_type
).write.mode(
    "overwrite"
).saveAsTable(
    "dtp_data.dtp_schema.gold_network_by_road_type"
)

spark.createDataFrame(
    gold_network_by_locality
).write.mode(
    "overwrite"
).saveAsTable(
    "dtp_data.dtp_schema.gold_network_by_locality"
)



# Network by road
spark.createDataFrame(
    gold_network_by_road
).write.mode(
    "overwrite"
).saveAsTable(
    "dtp_data.dtp_schema.gold_network_by_road"
)

spark.createDataFrame(
    gold_data_quality_summary
).write.mode(
    "overwrite"
).saveAsTable(
    "dtp_data.dtp_schema.gold_data_quality_summary"
)

spark.createDataFrame(
    gold_dq_by_locality
).write.mode(
    "overwrite"
).saveAsTable(
    "dtp_data.dtp_schema.gold_dq_by_locality"
)

spark.createDataFrame(
    gold_managed_roads_fact
).write.mode(
    "overwrite"
).saveAsTable(
    "dtp_data.dtp_schema.gold_managed_roads_fact"
)

print("Gold Delta tables created successfully")

Gold Delta tables created successfully


In [0]:
gold_tables = [
    "gold_network_overview",
    "gold_network_by_class",
    "gold_network_by_rma_class",
    "gold_network_by_road_type",
    "gold_network_by_locality",
    "gold_network_by_road",
    "gold_data_quality_summary",
    "gold_dq_by_locality",
    "gold_managed_roads_fact"
]

print("=== GOLD OUTPUT VERIFICATION ===")

for table in gold_tables:

    full_table_name = (
        f"dtp_data.dtp_schema.{table}"
    )

    row_count = spark.table(
        full_table_name
    ).count()

    print(
        f"{table:<35} {row_count:,} rows"
    )

print("\nGold output verification: PASS")

=== GOLD OUTPUT VERIFICATION ===
gold_network_overview               6 rows
gold_network_by_class               7 rows
gold_network_by_rma_class           5 rows
gold_network_by_road_type           25 rows
gold_network_by_locality            2,219 rows
gold_network_by_road                921 rows
gold_data_quality_summary           8 rows
gold_dq_by_locality                 2,219 rows
gold_managed_roads_fact             90,797 rows

Gold output verification: PASS


## Gold Layer Complete

The Silver DTP Managed Roads dataset has been transformed into
business-facing Asset Intelligence outputs.

### Gold Outputs

- Network overview KPIs
- Network length by road classification
- Network length by RMA class
- Network length by road type
- Network length by locality
- Road-level network summaries
- Overall data-quality metrics
- Locality-level quality exception summaries
- Detailed road-segment fact table for Power BI

The Gold layer converts cleaned road-segment data into measures that can
support stakeholder reporting, network understanding and data-quality
monitoring.

The Gold outputs are ready for Power BI dashboard development.